# AI Sommelier RAG: 이미지와 와인 리뷰로 추천하기

이 노트북은 검색 없는 추천과 RAG 추천을 비교한다. 요리 이미지를 검색 query로 바꾸고, 앞 노트북에 저장한 Wine Magazine 리뷰를 근거로 와인을 추천한다.

### 전체 흐름

`이미지 URL → 풍미 query → Retriever → list[Document] → context → 와인 추천`

### Baseline과 RAG의 차이

- Baseline: 모델의 사전학습 지식만 사용한다.
- RAG: Pinecone에서 찾은 실제 와인 리뷰를 Prompt의 근거로 사용한다.

### 사용하는 입력과 저장소

- Multimodal message: 텍스트와 이미지 URL을 한 메시지에 담는다.
- Query embedding: 이미지에서 만든 풍미 문장을 숫자 vector로 바꾼다.
- Pinecone index: 앞 노트북에서 만든 `winemag-review-data`를 사용한다.
- 기본 namespace: 별도 namespace를 지정하지 않은 레코드를 검색한다.

RAG의 인덱싱은 앞 노트북에서 완료했다. 여기서는 요청마다 Retrieval과 Generation을 실행한다. 이미지 해석은 Retrieval에 넣을 query를 만드는 전처리 단계이다.


## 패키지 준비

이 실습은 LangChain Runnable로 OpenAI 모델과 Pinecone Retriever를 연결한다.

- `langchain`: Prompt, Parser, Runnable을 제공한다.
- `langchain-openai`: 이미지 Chat Model과 query embedding을 제공한다.
- `langchain-pinecone`: Pinecone index를 LangChain Retriever로 연결한다.
- `pinecone`: Pinecone 서비스에 접속하는 공식 SDK이다.
- `python-dotenv`: `.env`의 API 설정을 환경 변수로 불러온다.
- `langchain-community`: 앞 인덱싱 단계의 `CSVLoader`와 같은 실습 환경을 유지한다.


In [1]:
from langchain_community.tools.azure_ai_services import image_analysis
from torch._C._dynamo.guards import TRUE_MATCH
%pip install -U langchain langchain-openai langchain-pinecone langchain-community pinecone python-dotenv


  Using cached pinecone-9.1.0-cp310-abi3-win_amd64.whl.metadata (6.3 kB)
Note: you may need to restart the kernel to use updated packages.


## API 설정 불러오기

`.env`의 API 키는 각 SDK의 인증에 사용하며 화면에 출력하지 않는다.

### 필요한 설정

- `OPENAI_API_KEY`: 이미지 해석, embedding, 추천 생성에 사용한다.
- `PINECONE_API_KEY`: `winemag-review-data` index를 검색할 때 사용한다.
- `OPENAI_EMBEDDING_MODEL`: 앞 인덱싱 단계에서 `text-embedding-3-small`로 설정한다.

### 앞 노트북과 같아야 하는 값

- embedding 모델: `text-embedding-3-small`이다.
- index 이름: `winemag-review-data`이다.
- namespace: 별도 값을 지정하지 않은 기본 namespace이다.

Pinecone index는 앞 인덱싱 노트북에서 생성·적재가 끝난 상태여야 한다.


In [2]:
import os

from dotenv import load_dotenv

load_dotenv()

CHAT_MODEL_NAME = os.getenv('OPENAI_CHAT_MODEL', 'gpt-5.6-luna')


## Baseline 1: 요리 이름으로 와인 추천하기

**Baseline**은 개선 방법의 효과를 비교하기 위한 기준 실행이다. 첫 번째 baseline은 외부 문서를 검색하지 않는다.

- 입력: 요리 이름 `도미회`이다.
- 처리: Prompt와 모델의 사전학습 지식만 사용한다.
- 출력: 와인 추천 문자열이다.
- 한계: 추천에 사용한 외부 리뷰와 출처가 없다.

LCEL의 `|`는 Runnable을 왼쪽에서 오른쪽으로 연결한다. 앞 단계의 출력이 다음 단계의 입력이 된다.


### Text Prompt → Chat Model → 문자열

세 Runnable을 `|`로 연결한다.

- `ChatPromptTemplate`: `system` 지시와 `{query}`를 역할별 message로 만든다.
- `ChatOpenAI`: message를 받아 `AIMessage`를 생성한다.
- `StrOutputParser`: `AIMessage`에서 답변 텍스트만 꺼내 `str`로 바꾼다.

입력과 출력은 `{'query': str} → PromptValue → AIMessage → str` 순서로 변한다.


In [4]:
from langchain_core.prompts import ChatPromptTemplate

from langchain_core.output_parsers import StrOutputParser

from langchain_openai import ChatOpenAI

# 1. Prompt 생성
text_recommendation_prompt = ChatPromptTemplate.from_messages([
    ('system', '''페르소나: 당신은 와인과 음식 페어링에 대한 열정을 지닌 지식 많고 경험 풍부한 소믈리에이다. 다양한 와인 산지, 포도 품종, 테이스팅 노트에 대한 깊은 이해를 갖고 있다. 친근하고 다가가기 쉬운 태도로 초보자부터 전문가까지 모두가 와인을 즐길 수 있도록 돕는다.

역할: 소믈리에로서 각종 요리에 완벽하게 어울리는 와인을 전문가 수준으로 추천한다. 이용자가 새로운 와인을 탐험하도록 안내하며 와인 테이스팅의 섬세함을 이해할 수 있게 지원한다. 적절한 와인을 매치해 식사 경험을 한층 더 풍성하게 만드는 것이 목표이다.

예시:

* 구운 마늘 버터 새우를 위해서는 샤르도네나 알바리뇨를 추천한다. 와인의 산도가 버터의 풍미와 기름진 맛을 깔끔하게 잡아준다.
* 가성비 좋은 와인을 찾는다면 프랑스 남부의 뮈스카데나나 스페인 리베라 델 두에로 지역의 템프라니요를 추천한다. 각각의 풍미 프로필과 어울리는 요리를 함께 설명한다.
* 와인 보관 방법을 묻는다면 적정 온도(12–14℃), 습도(60–70%), 빛 차단과 진동 방지 등의 실용적인 팁을 제공한다.'''),
    ('human', '''
다음 요리에 어울리는 와인을 추천해주세요. (한국말로 답변해주세요.)

요리명: {query}
'''),
])


# 2. baseline_llm 생성
baseline_llm = ChatOpenAI(
    model_name = CHAT_MODEL_NAME,
    temperature=1, # 답변 다양성 허용
    use_responses_api=True,
    reasoning_effort = 'none' # 추가 추론 허용 X
    # (none, low, medium, high)
)

# 3. AIMessage -> str 으로 변환하는 객체
output_parser = StrOutputParser()

# 3 개의 Runnable을 연결(chain)
text_baseline_chain = (
    text_recommendation_prompt
    | baseline_llm
    | output_parser
)

# 5. baseline 결과 확인
print(text_baseline_chain.invoke({'query': '돈까스'}))

돈까스에는 **산도가 좋고 탄닌이 너무 강하지 않은 와인**이 잘 어울립니다. 바삭한 튀김옷과 돼지고기의 기름기를 산도가 깔끔하게 씻어주고, 소스의 달콤새콤한 맛과도 조화를 이루기 때문입니다.

### 가장 추천하는 와인

**1. 스페인 카바 브뤼**
- 드라이한 스파클링 와인
- 풍부한 기포가 튀김옷의 느끼함을 정리해 줍니다.
- 돈까스 소스의 단맛과 산미에도 잘 어울려 가장 무난한 선택입니다.
- 차갑게 **6–8℃**로 즐기세요.

**2. 독일 리슬링 카비넷 또는 드라이 리슬링**
- 사과, 레몬, 복숭아 같은 향과 선명한 산도가 특징입니다.
- 특히 소스가 달콤하거나 겨자, 양배추 샐러드가 곁들여진 돈까스와 잘 맞습니다.
- 약간의 잔당이 있는 리슬링은 소스의 단맛을 부드럽게 받아줍니다.

**3. 뉴질랜드 소비뇽 블랑**
- 자몽과 허브 향, 상쾌한 산도가 있습니다.
- 로스카츠처럼 담백한 돈까스나 소스를 적게 찍어 먹는 스타일에 좋습니다.
- 레몬을 곁들인 샐러드와도 훌륭합니다.

### 레드 와인을 원한다면

**피노 누아**를 추천합니다.
- 가벼운 바디와 부드러운 탄닌, 체리와 라즈베리 향이 특징입니다.
- 돼지고기와 자연스럽게 어울리며, 튀김옷의 고소함을 압도하지 않습니다.
- 프랑스 부르고뉴보다 비교적 부담 없는 **칠레·뉴질랜드 피노 누아**도 좋은 선택입니다.

### 한 병만 고른다면
**카바 브뤼**가 가장 추천됩니다. 돈까스의 기름기, 바삭한 식감, 소스의 단짠 풍미를 모두 균형 있게 잡아주는 선택입니다.


## Baseline 2: 요리 이미지로 와인 추천하기

**Multimodal message**는 한 메시지에 텍스트와 이미지처럼 서로 다른 입력 형식을 함께 담는다.

- `text` block: 모델이 수행할 요청을 전달한다.
- `image_url` block: 모델이 읽을 공개 이미지 주소를 전달한다.
- 출력: 이미지에서 추정한 요리와 와인 추천 문자열이다.

이미지 URL은 모델 서버가 접근할 수 있어야 한다. 이 단계도 Wine Magazine 리뷰는 검색하지 않는다.


In [7]:
from langchain_core.prompts import HumanMessagePromptTemplate

# 1. Prompt 생성
# - HumanMessagePromptTemplate을 이용해서 MUltiModal 입력 만들기
image_recommendation_prompt = ChatPromptTemplate.from_messages([
    ('system', '''페르소나: 당신은 와인과 음식 페어링에 대한 열정을 지닌 지식 많고 경험 풍부한 소믈리에이다. 다양한 와인 산지, 포도 품종, 테이스팅 노트에 대한 깊은 이해를 갖고 있다. 친근하고 다가가기 쉬운 태도로 초보자부터 전문가까지 모두가 와인을 즐길 수 있도록 돕는다.

역할: 소믈리에로서 각종 요리에 완벽하게 어울리는 와인을 전문가 수준으로 추천한다. 이용자가 새로운 와인을 탐험하도록 안내하며 와인 테이스팅의 섬세함을 이해할 수 있게 지원한다. 적절한 와인을 매치해 식사 경험을 한층 더 풍성하게 만드는 것이 목표이다.

예시:

* 구운 마늘 버터 새우를 위해서는 샤르도네나 알바리뇨를 추천한다. 와인의 산도가 버터의 풍미와 기름진 맛을 깔끔하게 잡아준다.
* 가성비 좋은 와인을 찾는다면 프랑스 남부의 뮈스카데나나 스페인 리베라 델 두에로 지역의 템프라니요를 추천한다. 각각의 풍미 프로필과 어울리는 요리를 함께 설명한다.
* 와인 보관 방법을 묻는다면 적정 온도(12–14℃), 습도(60–70%), 빛 차단과 진동 방지 등의 실용적인 팁을 제공한다.'''),
    HumanMessagePromptTemplate.from_template([
        {'text': '다음 요리에 어울리는 와인을 추천해주세요. (한국말로 답변해주세요.)'},
        {'image_url': '{image_url}'},
    ]),
])

# 2. chain 구성
image_baseline_chain = (
    image_recommendation_prompt
    | baseline_llm
    | output_parser
)

# 3. chain.invoke()
# 이때, {'image_url}' 자리에 들어갈 이미지 주소 작성
image_url = 'https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcSTudOc575vygl6S0CBi7ROKSTW4bDwp8N3ChjutvqkXg&s=10'

print(image_baseline_chain.invoke({'image_url': image_url}))

사진 속 요리는 **바삭한 프라이드치킨**으로 보입니다. 튀김의 기름기를 씻어내고 바삭함을 살려주는 **높은 산도와 탄산, 지나치게 무겁지 않은 바디**의 와인이 잘 어울립니다.

### 가장 추천: 브뤼 스파클링 와인
- **프로세코 브뤼**: 사과·배의 산뜻한 과실 향과 탄산이 튀김옷의 기름기를 깔끔하게 정리합니다.
- **카바 브뤼**: 프로세코보다 드라이하고 산도가 뚜렷해 담백한 치킨과 특히 좋습니다.
- **샴페인 브뤼**: 예산이 허락한다면 최고의 선택입니다. 섬세한 기포와 고소한 빵 풍미가 치킨의 튀김옷과 훌륭하게 어울립니다.

### 가성비 좋은 화이트 와인
- **소비뇽 블랑**: 상쾌한 산도와 시트러스 풍미가 느끼함을 잡아줍니다. 매콤한 양념이나 피클을 곁들인 치킨에도 잘 맞습니다.
- **알바리뇨**: 레몬과 자몽 같은 산뜻함, 약간의 짭짤한 미네랄감이 있어 후라이드치킨을 깔끔하게 즐길 수 있습니다.
- **드라이 리슬링**: 산도가 뛰어나고 과실 풍미가 선명합니다. 치킨에 매콤달콤한 소스를 곁들인다면 약간의 잔당이 있는 리슬링도 좋습니다.

### 레드 와인을 원한다면
- **가메(보졸레)** 또는 **가벼운 피노 누아**를 추천합니다. 차갑게 살짝 칠링하면 튀김의 고소함을 해치지 않고 붉은 과실 풍미가 잘 어울립니다. 탄닌이 강한 카베르네 소비뇽이나 쉬라즈는 튀김과 다소 무겁게 느껴질 수 있습니다.

**한 병만 고른다면:**  
👉 **브뤼 프로세코 또는 카바 브뤼를 6–8℃로 차갑게 준비**해 보세요. 후라이드치킨의 바삭함과 가장 산뜻하게 어울립니다.


## Baseline 3: 와인 이미지로 요리 추천하기

세 번째 baseline은 입력과 출력의 방향을 바꾼다.

- 입력: 와인 라벨 이미지 URL이다.
- 처리: 라벨과 와인 특징을 multimodal 모델이 해석한다.
- 출력: 어울리는 요리 추천 문자열이다.

같은 Prompt → Model → Parser 구조에서도 system 역할과 human 질문을 바꾸면 Chain의 목적이 달라진다.


In [8]:
wine_image_prompt = ChatPromptTemplate.from_messages([
    ('system', '''페르소나(Persona): 당신은 와인과 음식의 조화를 깊이 이해하는 경험 많은 소믈리에이다. 다양한 와인 산지, 포도 품종, 테이스팅 노트에 대해 해박하며, 누구에게나 친근하고 쉽게 설명하는 능력을 가지고 있다.

역할(Role): 소믈리에로서, 당신의 역할은 특정 와인에 가장 잘 어울리는 요리를 전문적으로 추천하는 것이다. 와인의 향, 맛, 산도, 바디감 등을 분석해 최적의 음식 조합을 제안한다. 당신의 목표는 손님이 가진 와인을 더욱 특별하게 즐길 수 있도록, 완벽한 음식 페어링을 안내하는 것이다.

예시(Examples):

- 누군가가 ‘리슬링(Riesling)’ 와인을 가지고 있다고 하면, 와인의 상큼한 산도와 과일향에 어울리는 매콤한 아시아 요리나 스파이시 치킨을 추천하고 그 이유를 설명한다.
- ‘까베르네 소비뇽(Cabernet Sauvignon)’ 와인에 맞는 음식을 물어보면, 풍부한 탄닌과 바디감을 살려줄 스테이크나 구운 양고기와의 페어링을 안내한다.
- ‘스파클링 와인’에 잘 어울리는 간단한 핑거푸드나 해산물 요리 등을 추천하며, 와인의 청량감을 살리는 방법을 알려준다.
- 특정 와인을 활용한 요리 레시피를 제안하거나, 와인과 함께 먹으면 맛의 밸런스가 좋아지는 음식 스타일을 설명한다.'''),
    HumanMessagePromptTemplate.from_template([
        {'text': '다음 와인에 어울리는 요리를 추천해주세요. (한국말로 답변해주세요.)'},
        {'image_url': '{image_url}'},
    ]),
])

wine_to_food_chain = (
    wine_image_prompt
    | baseline_llm
    | output_parser
)

wine_image_url = "https://blog.kakaocdn.net/dna/bYUWAk/btstywD4JwU/AAAAAAAAAAAAAAAAAAAAAI9ZX1oUh5I3-sh7Lg4RIAOxFD8MWJKmHrpd1Oa6t1MQ/img.png?credential=yqXZFxpELC7KVnFOS48ylbz2pIh7yKj8&expires=1788188399&allow_ip=&allow_referer=&signature=elbyxYDvXR7CCtH61kkR9JKldiE%3D"

print(wine_to_food_chain.invoke({'image_url': wine_image_url}))

사진의 와인은 **테소 라 몬하(Teso La Monja) 2016**으로 보입니다. 스페인 토로(Toro) 지역의 고급 **템프라니요(티ンタ 데 토로)** 와인으로, 농축된 검은 과실 향과 묵직한 바디, 탄탄한 탄닌, 오크에서 오는 향신료·바닐라·코코아 풍미가 특징입니다.

### 가장 추천하는 요리
**숯불에 구운 소고기 스테이크**
- 등심, 채끝, 안심보다는 풍미가 진한 **립아이**나 **토마호크**가 특히 좋습니다.
- 고기의 육즙과 구운 향이 와인의 농축된 과실 풍미와 잘 어울리고, 탄닌은 지방감을 깔끔하게 정리해줍니다.
- 소스는 진한 크림소스보다는 **소금·후추**, 또는 **후추 소스와 레드와인 소스**가 적합합니다.

### 함께 잘 맞는 요리
1. **허브를 곁들인 양갈비 구이**  
   로즈마리와 마늘의 향이 와인의 스파이시한 오크 풍미와 조화를 이룹니다.

2. **소고기 갈비찜**  
   단맛을 과하게 내지 않고 간장과 배의 단맛을 절제하면 좋습니다. 부드러운 고기와 진한 양념이 와인의 깊이를 살려줍니다.

3. **숯불 닭다리 또는 오리구이**  
   껍질의 고소한 지방과 불향이 묵직한 레드 와인과 잘 맞습니다.

4. **버섯과 소고기를 넣은 라구 파스타**  
   토마토의 산미와 고기의 감칠맛이 와인의 과실감과 균형을 잡아줍니다.

5. **숙성 치즈**  
   만체고, 숙성 체다, 콩테처럼 단단하고 풍미가 깊은 치즈를 추천합니다.

### 피하면 좋은 음식
회, 담백한 흰살생선, 섬세한 샐러드처럼 맛이 가벼운 음식은 와인에 눌릴 수 있습니다. 또한 지나치게 매운 음식은 알코올감과 탄닌을 더 거칠게 느끼게 할 수 있습니다.

### 마시는 팁
- **16~18℃** 정도로 서빙하세요.
- 마시기 전 **1~2시간 디캔팅**하면 탄닌이 부드러워지고 검은 과실과 향신료 향이 더 잘 열립니다.
- 가장 이상적인 한 가지 조합은 **소금·후추로 간한 립아이 스테이크와 구운 버섯**입니다.


## Wine Magazine 기반 2-step RAG

RAG 본체는 Retrieval과 Generation 두 단계로 구성된다. 이미지 해석은 검색용 query를 만드는 전처리이다.

### 처리 순서

1. 이미지 해석: 이미지 URL을 풍미 query 문자열로 바꾼다.
2. Retrieval: query와 가까운 리뷰를 `list[Document]`로 가져온다.
3. Context 구성: 검색된 `Document.page_content`를 문자열 하나로 묶는다.
4. Generation: 풍미와 context를 사용해 와인을 추천한다.

`이미지 URL → 풍미 str → list[Document] → context str → 추천 str`


### LCEL에서 사용할 Runnable

`Runnable`은 입력을 받아 한 작업을 수행하고 출력을 반환하는 공통 실행 단위이다. `|`로 여러 Runnable을 연결하면 `RunnableSequence`가 만들어진다.

- `RunnableLambda`: 일반 Python 함수를 LCEL 단계로 사용한다.
- `RunnableParallel`: 같은 입력을 여러 분기에 전달하고 결과를 dict로 묶는다.
- `RunnablePassthrough`: 입력값을 바꾸지 않고 그대로 반환한다.

Retrieval 단계는 같은 풍미 문자열을 두 곳에서 사용한다.

- 보존 분기: `dish_flavor`에 원래 문자열을 남긴다.
- 검색 분기: Retriever가 `retrieved_documents`를 만든다.
- 병합 결과: `{'dish_flavor': str, 'retrieved_documents': list[Document]}`이다.


### 이미지 URL을 검색 query로 바꾸기

Pinecone에는 텍스트 와인 리뷰가 저장되어 있다. 따라서 요리 이미지를 바로 비교하지 않고 텍스트 풍미로 변환한다.

- 입력: `{'image_urls': list[str]}`이다.
- 변환: 각 URL을 multimodal image block으로 만든다.
- 출력: 검색에 사용할 영어 한 문장이다.
- 다음 사용처: Pinecone Retriever의 query로 전달한다.

`describe_dish_flavor()`는 결과 문자열을 직접 반환하지 않는다. 이미지 Prompt → Model → Parser를 연결한 Runnable을 반환한다.


In [10]:
def describe_dish_flavor(query: dict):


    # 1. 음식 이미지 -> 영어 한 문장으로 변환 요청 prompt 생성
    dish_flavor_prompt = ChatPromptTemplate.from_messages([
            ('system', '''페르소나: 당신은 조리 기법, 풍미 특성과 식재료 조합을 깊이 이해하는 뛰어난 음식 전문가이다. 다양한 요리를 탐구하는 데 열정이 있으며 음식의 감각적 경험을 구체적으로 표현할 수 있다. 실무 경험과 이론 지식을 모두 갖추고 있어 신뢰할 수 있는 분석을 제공한다.

    역할: 음식 전문가로서 다양한 요리의 맛, 식감과 향을 분석한다. 식재료와 조리 방법을 구체적으로 평가하고, 균형 있고 조화로운 요리를 만드는 원리를 설명한다. 또한 요리 기술을 향상하고 미식의 가치를 이해할 수 있도록 돕는다.

    예시:

    요리의 풍미 특성을 분석할 때는 산미, 단맛, 쓴맛과 감칠맛의 균형을 설명하고 이 요소들이 어떻게 어우러져 복합적인 맛을 만드는지 분석한다.
    특정 식재료의 풍미를 살리는 방법을 묻는다면 양파를 캐러멜화해 깊은 맛을 내거나 고기의 본연의 맛을 살리도록 적절히 간하는 방법처럼 실용적인 조언을 제공한다.
    음식 조합을 설명할 때는 해산물에 감귤류를 곁들여 산뜻함을 더하거나 허브로 전체 풍미를 끌어올리는 사례처럼 서로 보완하는 식재료와 풍미를 제안하고 그 이유를 설명한다.'''),
            ('human', '''
    이미지를 바탕으로 요리를 분석한다. 와인 리뷰 검색 query로 사용할 수 있도록 핵심 재료, 조리법과 풍미를 영어 한 문장으로 간결하게 출력한다.
    '''),
        ])

    # 2. image_url의 각 문자열을 multimodal image content block 으로 변경
    image_contents = [
        {'image_url' : image_url}
        for image_url in query.get('image_url', [])
    ]

    # 3. HumanMessage로 변환하여 Prompt에 추가
    dish_flavor_prompt += HumanMessagePromptTemplate.from_template(image_contents)

    # 4. 이미지를 영어 한 문장으로 변환할 llm 준비
    image_analysis_llm = ChatOpenAI(
        model = CHAT_MODEL_NAME,
        use_responses_api=True,
        temperature=0,
        reasoning_effort='none'
    )

    # 5. chain 구성 후 반환
    return dish_flavor_prompt | image_analysis_llm | StrOutputParser()

### 이미지 해석 Runnable 실행하기

`RunnableLambda`는 일반 Python 함수를 `invoke()`로 실행할 수 있게 감싼다.

이 예제에는 한 단계가 더 있다.

1. `RunnableLambda`가 `describe_dish_flavor(payload)`를 호출한다.
2. 함수가 Prompt → Model → Parser Runnable을 반환한다.
3. LangChain이 반환된 Runnable에도 같은 `payload`를 전달해 이어서 실행한다.
4. 최종 결과로 풍미 문자열을 반환한다.

출력은 음식·조리법·풍미가 포함된 검색용 영어 한 문장이어야 한다.


In [14]:
from langchain_core.runnables import RunnableLambda

# 1. 일반 함수 describe_dish_flavor를
# LCEL에서 실행할 수 있도록 RunnableLambda로 감싸기
describe_dish_flavor_chain = RunnableLambda(describe_dish_flavor)

# 2. 이미지 주소를 전달하여 결과 확인
dish_flavor = describe_dish_flavor_chain.invoke({
    'image_url':['https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcRV7oCQmx631YgDGlLRaLAz3jLyjk1UNOmJQlptPOEuMA&s=10']
})

print(dish_flavor)

Korean-style grilled pork belly with kimchi and zucchini, seared until crisp and caramelized, offering rich savory, smoky, fatty flavors balanced by spicy tang and fresh vegetal notes.


### Pinecone index를 Retriever로 연결하기

앞 노트북은 와인 리뷰를 `text-embedding-3-small`로 변환해 `winemag-review-data` index의 기본 namespace에 저장했다.

### 검색 설정

- embedding 모델: 문서 인덱싱과 같은 `text-embedding-3-small`이다.
- index: `winemag-review-data`이다.
- namespace: 별도 값을 지정하지 않은 기본 namespace이다.
- 검색 방식: 의미 유사도 검색이다.
- `k=5`: 상위 리뷰를 최대 다섯 개 반환한다.

`as_retriever()`는 Vector Store를 `query str → list[Document]` 구조의 검색 인터페이스로 바꾼다.


In [ ]:
from langchain_openai import OpenAIEmbeddings

from langchain_pinecone import PineconeVectorStore


### 대표 query로 검색 결과 확인하기

이미지 해석 단계와 같은 형식의 영어 풍미 문장을 Retriever에 전달한다.

- 입력: 음식의 재료·조리법·풍미를 설명한 문자열이다.
- 출력: 유사도 순서의 `list[Document]`이다.
- 확인할 값: 각 문서의 `page_content`와 `metadata`이다.

검색 결과가 존재한다고 항상 관련성이 높은 것은 아니다. 생성 전에 음식 풍미와 실제 와인 리뷰가 연결되는지 읽어 본다.


### 검색 결과를 Generation 입력으로 바꾸기

Generation Prompt에는 원래 풍미와 검색 리뷰가 모두 필요하다.

### 두 분기의 역할

- `dish_flavor`: `RunnablePassthrough`가 입력 문자열을 그대로 보존한다.
- `retrieved_documents`: Retriever가 `list[Document]`를 반환한다.

### 결과 변환

1. `RunnableParallel`이 두 결과를 dict로 묶는다.
2. `format_documents()`가 각 `page_content`를 구분선으로 연결한다.
3. `build_retrieval_context()`가 Prompt 변수 이름에 맞춘 dict를 반환한다.

최종 구조는 `{'dish_flavor': str, 'wine_reviews': str}`이다.


In [ ]:
from langchain_core.documents import Document

from langchain_core.runnables import RunnableParallel, RunnablePassthrough


### Retrieval Runnable의 출력 확인하기

Generation에 연결하기 전에 반환 dict의 구조를 확인한다.

- `dish_flavor`: 처음 입력한 풍미 query 문자열이다.
- `wine_reviews`: 검색된 리뷰 본문을 합친 context 문자열이다.

두 key 이름은 다음 `ChatPromptTemplate`의 `{dish_flavor}`, `{wine_reviews}`와 정확히 같아야 한다.


### 검색 근거로 추천 Runnable 만들기

Generation은 Retrieval이 만든 dict를 Prompt에 채워 최종 추천을 생성한다.

- 입력: `{'dish_flavor': str, 'wine_reviews': str}`이다.
- Prompt: 요리 풍미와 검색 리뷰를 각각 지정된 위치에 넣는다.
- Model: 리뷰를 읽고 한국어 추천을 생성한다.
- Parser: `AIMessage`를 추천 문자열로 바꾼다.

`recommend_wines()`도 결과 문자열이 아니라 Prompt → Model → Parser Runnable을 반환한다. 뒤의 `RunnableLambda`가 반환된 Runnable을 같은 입력 dict로 이어서 실행한다.

프롬프트 지시만으로 사실 일치가 완전히 보장되지는 않는다. 추천 이름과 이유가 실제 `wine_reviews`에 있는지 별도로 확인한다.


In [ ]:
recommend_wines_prompt = ChatPromptTemplate.from_messages([
        ('system', '''페르소나: 당신은 와인과 음식 페어링에 열정을 지닌 지식 많고 경험 풍부한 소믈리에이다. 다양한 와인 산지, 포도 품종과 테이스팅 노트를 폭넓게 이해한다. 친근하고 다가가기 쉬운 태도로 초보자와 애호가 모두가 와인을 편하게 접할 수 있도록 돕는다.

역할: 소믈리에로서 다양한 요리에 잘 어울리는 와인을 전문적으로 추천한다. 이용자가 새로운 와인을 탐색하도록 안내하고 와인 테이스팅의 섬세한 차이를 이해하도록 돕는다. 적절한 와인과 요리를 연결해 식사 경험을 향상하는 것이 목표이다.

예시:

구운 마늘 버터 새우에 어울리는 와인을 묻는다면 Chardonnay 또는 Albariño를 제안하고, 와인의 산미가 요리의 풍부한 맛과 기름진 느낌을 어떻게 균형 있게 잡아 주는지 설명한다.
가격이 합리적이면서 품질 좋은 와인을 묻는다면 여러 산지의 구체적인 선택지를 추천하고 각 와인의 풍미 특성과 어울리는 음식을 설명한다.
와인 보관법을 설명할 때는 와인의 품질을 유지할 수 있는 온도, 습도와 적절한 보관 조건을 실용적으로 안내한다.'''),
        ('human', '''
와인 페어링 추천해주세요.
아래의 요리설명과 와인리뷰만을 참고하여 한글로 답변해주세요.
아래의 요리설명과 와인리뷰외의 내용을 추가하지 말아주세요.

요리설명:
{dish_flavor}

와인리뷰:
{wine_reviews}

추천 와인과 이유:
'''),
    ])


### 전체 이미지 RAG Chain 연결하기

세 Runnable을 `|`로 연결한다. 각 단계의 출력 자료형이 다음 단계의 입력 자료형과 이어져야 한다.

1. `describe_dish_flavor_chain`: `{'image_urls': list[str]}` → 풍미 `str`이다.
2. `wine_review_retrieval_chain`: 풍미 `str` → `{dish_flavor, wine_reviews}` dict이다.
3. `recommend_wines_chain`: 검색 context dict → 추천 `str`이다.

이미지 해석은 query 전처리이고, Retrieval과 Generation이 2-step RAG 본체이다. 최종 Chain은 추천 문자열만 반환하므로 검색 근거는 앞의 Retrieval 출력 확인 단계에서 별도로 검토한다.
